# 00 · Primeiro contato — Is My Game Relevant?

Entrega 1: carregar o dataset, definir o alvo, separar o teste e medir o baseline.

**Alvo:** faixa de relevância derivada da coluna `Estimated owners` (estimativa de donos do SteamSpy).
**Métrica principal:** F1-macro. **Secundária:** precisão da classe `alta`.

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path(os.environ.get('DATA_PATH', '../data/games.csv'))
TEST_IDS_PATH = DATA_PATH.parent / 'test_appids.csv'

## 1. Carregamento

O `games.csv` do Kaggle tem um cabeçalho com `DiscountDLC count` grudado (duas colunas num nome só). Sem correção, o pandas desloca todas as colunas uma posição. A função abaixo separa o nome e normaliza tudo para `snake_case`.

In [3]:
def carregar_games(path):
    with open(path, encoding='utf-8') as f:
        header = f.readline().strip().split(',')
    nomes = []
    for col in header:
        nomes.extend(['Discount', 'DLC count'] if col == 'DiscountDLC count' else [col])
    df = pd.read_csv(path, header=None, skiprows=1, names=nomes, index_col=False, low_memory=False)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    return df

df = carregar_games(DATA_PATH)
print(df.shape)
df.info()

(125855, 40)
<class 'pandas.DataFrame'>
RangeIndex: 125855 entries, 0 to 125854
Data columns (total 40 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   appid                       125855 non-null  int64  
 1   name                        125854 non-null  str    
 2   release_date                125855 non-null  str    
 3   estimated_owners            125855 non-null  str    
 4   peak_ccu                    125855 non-null  int64  
 5   required_age                125855 non-null  int64  
 6   price                       125855 non-null  float64
 7   discount                    125855 non-null  int64  
 8   dlc_count                   125855 non-null  int64  
 9   about_the_game              117393 non-null  str    
 10  supported_languages         125855 non-null  str    
 11  full_audio_languages        125855 non-null  str    
 12  reviews                     12181 non-null   str    
 13  header_image

In [ ]:
df[['appid', 'name', 'release_date', 'estimated_owners', 'price', 'genres']].head()

## 2. Alvo

`estimated_owners` vem em faixas do SteamSpy (`"0 - 20000"`, `"20000 - 50000"`...). Elas são agrupadas em três classes:

| Classe | Donos estimados |
|---|---|
| baixa | menos de 20 mil |
| media | 20 mil a 200 mil |
| alta | 200 mil ou mais |

Dois filtros evitam rótulos falsos:
- `"0 - 0"` significa que o SteamSpy não tem dado, não que o jogo tem zero donos;
- jogos lançados nos últimos 12 meses da coleta ainda não acumularam donos, então seriam rotulados como `baixa` injustamente.

In [4]:
faixa = df['estimated_owners'].astype(str).str.replace(',', '').str.split(' - ', expand=True)
df['owners_min'] = pd.to_numeric(faixa[0], errors='coerce')
df['owners_max'] = pd.to_numeric(faixa[1], errors='coerce')
df['data_lancamento'] = pd.to_datetime(df['release_date'], format='mixed', errors='coerce')

data_coleta = df['data_lancamento'].max()
corte = data_coleta - pd.DateOffset(months=12)

antes = len(df)
df = df[(df['owners_max'] > 0) & df['data_lancamento'].notna() & (df['data_lancamento'] <= corte)].copy()
print(f'data mais recente no dataset: {data_coleta.date()} | corte: {corte.date()}')
print(f'linhas: {antes} -> {len(df)}')

df['relevancia'] = pd.cut(df['owners_min'], bins=[-1, 19_999, 199_999, np.inf],
                          labels=['baixa', 'media', 'alta']).astype(str)
df['relevancia'].value_counts(normalize=True).round(4)

data mais recente no dataset: 2026-12-01 | corte: 2025-12-01
linhas: 125855 -> 100338


relevancia
baixa    0.7453
media    0.2013
alta     0.0534
Name: proportion, dtype: float64

## 3. Colunas de entrada

Só entra o que um desenvolvedor sabe **antes** de lançar (teste do formulário). Tudo que é consequência do lançamento fica de fora.

In [5]:
POS_LANCAMENTO = ['estimated_owners', 'owners_min', 'owners_max', 'peak_ccu', 'positive', 'negative',
                  'recommendations', 'user_score', 'score_rank', 'metacritic_score', 'metacritic_url',
                  'reviews', 'average_playtime_forever', 'average_playtime_two_weeks',
                  'median_playtime_forever', 'median_playtime_two_weeks', 'discount', 'dlc_count', 'notes']
SEM_INFORMACAO = ['appid', 'name', 'about_the_game', 'header_image', 'website', 'support_url',
                  'support_email', 'screenshots', 'movies']

ENTRADAS = ['release_date', 'required_age', 'price', 'supported_languages', 'full_audio_languages',
            'windows', 'mac', 'linux', 'achievements', 'developers', 'publishers',
            'categories', 'genres', 'tags']

print('fora do X por vazamento:', [c for c in POS_LANCAMENTO if c in df.columns])
print('colunas não classificadas:', [c for c in df.columns
      if c not in POS_LANCAMENTO + SEM_INFORMACAO + ENTRADAS + ['data_lancamento', 'relevancia']])
df[ENTRADAS].isna().mean().sort_values(ascending=False).round(3)

fora do X por vazamento: ['estimated_owners', 'owners_min', 'owners_max', 'peak_ccu', 'positive', 'negative', 'recommendations', 'user_score', 'score_rank', 'metacritic_score', 'metacritic_url', 'reviews', 'average_playtime_forever', 'average_playtime_two_weeks', 'median_playtime_forever', 'median_playtime_two_weeks', 'discount', 'dlc_count', 'notes']
colunas não classificadas: []


tags                    0.170
categories              0.008
publishers              0.005
genres                  0.001
developers              0.000
release_date            0.000
windows                 0.000
full_audio_languages    0.000
supported_languages     0.000
price                   0.000
required_age            0.000
achievements            0.000
mac                     0.000
linux                   0.000
dtype: float64

## 4. Divisão treino/teste

80/20 estratificado. Os `appid` do teste são salvos em `data/test_appids.csv` para que os próximos notebooks usem exatamente a mesma separação. O teste só é aberto na Entrega 3.

In [6]:
X = df[ENTRADAS]
y = df['relevancia']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
df.loc[X_te.index, ['appid']].to_csv(TEST_IDS_PATH, index=False)

print(X_tr.shape, X_te.shape)
pd.concat([y_tr.value_counts(normalize=True), y_te.value_counts(normalize=True)],
          axis=1, keys=['treino', 'teste']).round(4)

(80270, 14) (20068, 14)


,treino,teste
relevancia,,
baixa,0.7453,0.7453
media,0.2013,0.2013
alta,0.0534,0.0534


## 5. Baseline

Dois Dummies, avaliados com validação cruzada estratificada (5 dobras) só no treino:
- `most_frequent`: sempre responde a classe majoritária;
- `stratified`: sorteia classes na proporção do treino.

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = {
    'f1_macro': 'f1_macro',
    'acuracia': 'accuracy',
    'precisao_alta': make_scorer(precision_score, labels=['alta'], average='macro', zero_division=0),
    'recall_alta': make_scorer(recall_score, labels=['alta'], average='macro', zero_division=0),
}

def avaliar(modelo, X, y):
    r = cross_validate(modelo, X, y, cv=cv, scoring=SCORING)
    return {k: round(r[f'test_{k}'].mean(), 4) for k in SCORING}

resultados = {
    'dummy_most_frequent': avaliar(DummyClassifier(strategy='most_frequent'), X_tr, y_tr),
    'dummy_stratified': avaliar(DummyClassifier(strategy='stratified', random_state=RANDOM_STATE), X_tr, y_tr),
}
pd.DataFrame(resultados).T

,f1_macro,acuracia,precisao_alta,recall_alta
dummy_most_frequent,0.2847,0.7453,0.0000,0.0000
dummy_stratified,0.3320,0.5987,0.0518,0.0511


## 6. Sinal preliminar (opcional)

Não é o modelo da Entrega 3. É uma regressão logística com poucas colunas, só para ver se existe sinal acima do baseline antes de investir na EDA. Tudo dentro de um `Pipeline`, então cada dobra aprende imputação e encoding só com os seus dados de treino.

In [8]:
def features_simples(X):
    out = pd.DataFrame(index=X.index)
    out['price'] = pd.to_numeric(X['price'], errors='coerce')
    out['achievements'] = pd.to_numeric(X['achievements'], errors='coerce')
    out['n_idiomas'] = X['supported_languages'].fillna('').str.count(',') + X['supported_languages'].notna()
    out['tem_pt_br'] = X['supported_languages'].fillna('').str.contains('Brazil').astype(int)
    for p in ['windows', 'mac', 'linux']:
        out[p] = X[p].astype(str).str.lower().eq('true').astype(int)
    out['mes'] = pd.to_datetime(X['release_date'], format='mixed', errors='coerce').dt.month.astype(str)
    out['genero_principal'] = X['genres'].fillna('sem_genero').str.split(',').str[0]
    return out

F_tr = features_simples(X_tr)
NUM = ['price', 'achievements', 'n_idiomas', 'tem_pt_br', 'windows', 'mac', 'linux']
CAT = ['mes', 'genero_principal']

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('esc', StandardScaler())]), NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=50), CAT),
])
pipe = Pipeline([('pre', pre), ('modelo', LogisticRegression(max_iter=2000, class_weight='balanced'))])

resultados['logreg_preliminar'] = avaliar(pipe, F_tr, y_tr)
pd.DataFrame(resultados).T

,f1_macro,acuracia,precisao_alta,recall_alta
dummy_most_frequent,0.2847,0.7453,0.0000,0.0000
dummy_stratified,0.3320,0.5987,0.0518,0.0511
logreg_preliminar,0.3962,0.5530,0.1319,0.5289


## 7. Para o README

Copiar a tabela acima para a seção **Métrica e baseline** do README. O número que a Entrega 3 precisa bater é o F1-macro do `dummy_most_frequent`.